# Teste manual dos providers

Chama cada fonte de dados em [`src/providers/`](../src/providers/) para validar credenciais, rede e paths locais.

**Pré-requisitos**

1. Na raiz do projeto: `pip install -e ".[dev]"`
2. Copiar [`.env.example`](../.env.example) → `.env` e preencher (mínimo: `ANBIMA_*` para API ANBIMA; `UPTODATA_*` para ajustes BMF)
3. Kernel apontando para o mesmo venv

Abra este notebook na pasta `notebooks/` **ou** na raiz do repositório.

In [1]:
import sys
from datetime import date, timedelta
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if (ROOT / "src" / "config").is_dir():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "src" / "config").is_dir():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("Execute na raiz do repo ou em notebooks/")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[misc, assignment]

from config import get_settings

get_settings.cache_clear()
settings = get_settings()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", settings.data_root)
print("LOG_LEVEL:", settings.log_level)
print("ANBIMA client_id definido:", bool(settings.anbima.client_id))
print("BCB CDI series_id:", settings.bcb.cdi_series_id)
print("BCB PTAX moeda_code (USD):", settings.bcb.ptax_moeda_code)
print("UpToData pasta IR:", settings.uptodata.pasta_interest_rate_base or "(vazio)")

PROJECT_ROOT: \\BRZ1WS2478\Brazil_Dept\Mesa_Papel\Chila\projetos\brazil_fixed_income_analytics
DATA_ROOT: \\BRZ1WS2478\Brazil_Dept\Mesa_Papel\Chila\projetos\brazil_fixed_income_analytics\data
LOG_LEVEL: INFO
ANBIMA client_id definido: True
BCB CDI series_id: 11
BCB PTAX moeda_code (USD): 61
UpToData pasta IR: X:\Interest_Rate\SettlementPrice


In [2]:
# Ajuste as datas de teste aqui
REF_DATE = (date.today() - timedelta(days=5)).isoformat()  # dia útil recente
DATES = [REF_DATE]
MES, ANO = int(REF_DATE[5:7]), int(REF_DATE[:4])

print("REF_DATE:", REF_DATE)
print("DATES:", DATES)


def preview_df(df: pd.DataFrame, title: str, n: int = 5) -> None:
    print("\n" + "=" * 60)
    print(f"{title} | shape={df.shape}")
    print("=" * 60)
    if df.empty:
        print("(DataFrame vazio)")
        return
    display(df.head(n))


def preview_list(items: list, title: str, n: int = 10) -> None:
    print("\n" + "=" * 60)
    print(f"{title} | len={len(items)}")
    print("=" * 60)
    print(items[:n])

REF_DATE: 2026-05-15
DATES: ['2026-05-15']


## 1. ANBIMA API (OAuth)

Requer `ANBIMA_CLIENT_ID` e `ANBIMA_CLIENT_SECRET` no `.env`.

In [4]:
from providers.anbima import AnbimaClient, MERCADO_SECUNDARIO_TPF, PROJECOES

client = AnbimaClient()

payload_ms = client.fetch_by_date(MERCADO_SECUNDARIO_TPF, REF_DATE)
print("mercado_secundario:", "OK" if payload_ms is not None else "sem dado (404 ou vazio)")
if payload_ms is not None:
    if isinstance(payload_ms, list):
        print("registros:", len(payload_ms))
        display(payload_ms[:2])
    else:
        display(payload_ms)

payload_proj = client.fetch_projecoes("04", "2026")
print("\nprojecoes:", "OK" if payload_proj is not None else "sem dado")
if payload_proj is not None:
    display(payload_proj if not isinstance(payload_proj, list) else payload_proj[:3])

mercado_secundario: OK
registros: 51


[{'tipo_titulo': 'LTN',
  'expressao': 'Taxa (% a.a.)/252',
  'data_vencimento': '2026-07-01',
  'data_referencia': '2026-05-15',
  'codigo_selic': '100000',
  'data_base': '2023-01-06',
  'taxa_compra': 14.4154,
  'taxa_venda': 14.3887,
  'taxa_indicativa': 14.4082,
  'intervalo_min_d0': 14.2764,
  'intervalo_max_d0': 14.5871,
  'intervalo_min_d1': 14.286,
  'intervalo_max_d1': 14.6007,
  'pu': 983.052856,
  'desvio_padrao': 0.00787399957264,
  'codigo_isin': 'BRSTNCLTN848'},
 {'tipo_titulo': 'NTN-B',
  'expressao': 'Taxa (% a.a.)/252',
  'data_vencimento': '2026-08-15',
  'data_referencia': '2026-05-15',
  'codigo_selic': '760199',
  'data_base': '2000-07-15',
  'taxa_compra': 10.2182,
  'taxa_venda': 10.1896,
  'taxa_indicativa': 10.2013,
  'intervalo_min_d0': 9.4063,
  'intervalo_max_d0': 10.9458,
  'intervalo_min_d1': 9.5887,
  'intervalo_max_d1': 11.1407,
  'pu': 4723.795827,
  'desvio_padrao': 0.00449065696641,
  'codigo_isin': 'BRSTNCNTB4U6'}]


projecoes: OK


[{'indice': 'IPCA',
  'tipo_projecao': 'PROJEÇÕES PARA O MÊS CORRENTE',
  'data_coleta': '2026-04-10',
  'mes_referencia': '04/2026',
  'variacao_projetada': 0.68,
  'data_validade': '2026-04-16'},
 {'indice': 'IPCA',
  'tipo_projecao': 'PROJEÇÕES PARA O MÊS POSTERIOR',
  'data_coleta': '2026-04-10',
  'mes_referencia': '05/2026',
  'variacao_projetada': 0.41},
 {'indice': 'IGP-M',
  'tipo_projecao': 'PROJEÇÕES PARA O MÊS CORRENTE',
  'data_coleta': '2026-04-10',
  'mes_referencia': '04/2026',
  'variacao_projetada': 1.61,
  'data_validade': '2026-04-13'}]

### Estimativa SELIC (feed ANBIMA)

Estimativa diária da taxa SELIC (% a.a./252). Mesmas credenciais OAuth da §1.

In [4]:
from providers.anbima import ESTIMATIVA_SELIC, fetch_estimativa_selic

print("URL:", f"{ESTIMATIVA_SELIC}?data={REF_DATE}")

df_est_selic = fetch_estimativa_selic("2026-04-13")
preview_df(df_est_selic, "ANBIMA — Estimativa SELIC")

df_est_selic_latest = fetch_estimativa_selic()
if not df_est_selic_latest.empty:
    preview_df(df_est_selic_latest, "ANBIMA — Estimativa SELIC (latest)")

URL: https://api.anbima.com.br/feed/precos-indices/v1/titulos-publicos/estimativa-selic?data=2026-05-15

ANBIMA — Estimativa SELIC | shape=(1, 2)


,data_referencia,estimativa_taxa_selic
0,2026-04-13,14.65



ANBIMA — Estimativa SELIC (latest) | shape=(1, 2)


,data_referencia,estimativa_taxa_selic
0,2026-05-20,14.4


In [11]:
payload_proj = client.fetch_projecoes("04", "2026")
pd.DataFrame(payload_proj)

,indice,tipo_projecao,data_coleta,mes_referencia,variacao_projetada,data_validade
0,IPCA,PROJEÇÕES PARA O MÊS CORRENTE,2026-04-10,04/2026,0.68,2026-04-16
1,IPCA,PROJEÇÕES PARA O MÊS POSTERIOR,2026-04-10,05/2026,0.41,NaN
2,IGP-M,PROJEÇÕES PARA O MÊS CORRENTE,2026-04-10,04/2026,1.61,2026-04-13
3,IGP-M,PROJEÇÕES PARA O MÊS POSTERIOR,2026-04-10,05/2026,0.82,NaN
4,IGP-M,PROJEÇÕES PARA O MÊS CORRENTE,2026-04-17,04/2026,2.68,2026-04-20
5,IGP-M,PROJEÇÕES PARA O MÊS POSTERIOR,2026-04-17,05/2026,0.89,NaN
6,IPCA,PROJEÇÕES PARA O MÊS CORRENTE,2026-04-28,04/2026,0.67,2026-04-29
7,IPCA,PROJEÇÕES PARA O MÊS POSTERIOR,2026-04-28,05/2026,0.45,NaN
8,IGP-M,PROJEÇÕES PARA O MÊS CORRENTE,2026-04-29,05/2026,0.83,2026-05-05
9,IGP-M,PROJEÇÕES PARA O MÊS POSTERIOR,2026-04-29,06/2026,0.44,NaN


## 2. Feriados (XLS público ANBIMA)

Sem credenciais. URL em `FERIADOS_XLS_URL`.

In [4]:
from providers.feriados import fetch_feriados

feriados = fetch_feriados()
preview_list(feriados, "Feriados nacionais (ISO)")


Feriados nacionais (ISO) | len=1264
['2001-01-01', '2001-02-26', '2001-02-27', '2001-04-13', '2001-04-21', '2001-05-01', '2001-06-14', '2001-09-07', '2001-10-12', '2001-11-02']


## 3. BCB — liquidações (ZIP NegE)

Download público; pode demorar conforme a rede.

In [5]:
from providers.bcb import build_negociacoes_url, fetch_negociacoes_bruto_por_datas

print("URL:", build_negociacoes_url(REF_DATE))

df_bcb = fetch_negociacoes_bruto_por_datas(DATES, date_column="DATA MOV")
preview_df(df_bcb, "BCB NegE (bruto)")

URL: https://www4.bcb.gov.br/pom/demab/negociacoes/download/NegE202605.ZIP

BCB NegE (bruto) | shape=(57, 19)


,DATA MOV,SIGLA,CODIGO,CODIGO ISIN,EMISSAO,VENCIMENTO,NUM DE OPER,QUANT NEGOCIADA,VALOR NEGOCIADO,PU MIN,PU MED,PU MAX,PU LASTRO,VALOR PAR,TAXA MIN,TAXA MED,TAXA MAX,NUM OPER COM CORRETAGEM,QUANT NEG COM CORRETAGEM
0,2026-05-13,LFT,210100,BRSTNCLF1RF7,13/03/2020,01/09/2026,13,17941,NaN,"18967,25409600","18987,63561000","18987,86441100","18983,18430136","18987,80744800","-0,0006","0,0032","0,3506",1,76
1,2026-05-13,LFT,210100,BRSTNCLF1RG5,03/07/2020,01/03/2027,100,128197,NaN,"18983,30733700","18986,97114200","18987,25680100","18975,08672802","18987,80744800","0,0037","0,0056","0,0301",9,414
2,2026-05-13,LFT,210100,BRSTNCLF1RH3,02/07/2021,01/09/2027,87,57306,NaN,"18981,61742200","18985,75947000","18987,80744800","18966,08068808","18987,80744800","0,0000","0,0082","0,0251",10,1122
3,2026-05-13,LFT,210100,BRSTNCLF1RI1,05/01/2022,01/03/2028,75,77549,NaN,"18979,66167800","18980,94596600","18982,01616600","18953,79727553","18987,80744800","0,0170","0,0201","0,0239",10,2794
4,2026-05-13,LFT,210100,BRSTNCLF1RK7,06/04/2022,01/09/2028,80,63071,NaN,"18879,00149000","18973,83242600","18978,31354400","18938,95126245","18987,80744800","0,0217","0,0319","0,2500",6,4457


## 4. BCB SGS — CDI diário (série 11)

API JSON pública (`api.bcb.gov.br`); não exige credenciais. Colunas retornadas: `data`, `valor` (% a.a.). Intervalos longos são buscados em chunks de ~10 anos.

In [2]:
from providers import build_sgs_url, fetch_cdi_daily

cfg_bcb = settings.bcb
print("CDI series_id:", cfg_bcb.cdi_series_id)
print("SGS URL (1 dia):", build_sgs_url(cfg_bcb.cdi_series_id, REF_DATE, REF_DATE, settings=cfg_bcb))

df_cdi = fetch_cdi_daily(start_date=REF_DATE, end_date=REF_DATE, settings=cfg_bcb)
preview_df(df_cdi, "BCB SGS — CDI diário")

CDI series_id: 11


NameError: name 'REF_DATE' is not defined

## 5. BCB PTAX — USD fechamento

CSV público em `ptax.bcb.gov.br` (`ChkMoeda=61`). Colunas: `data`, `taxa_compra`, `taxa_venda`, etc.

In [3]:
from providers import build_ptax_fechamento_url, fetch_ptax_usd

cfg_bcb = settings.bcb
print("PTAX moeda_code:", cfg_bcb.ptax_moeda_code)
print("URL (1 dia):", build_ptax_fechamento_url(REF_DATE, REF_DATE, settings=cfg_bcb))

df_ptax = fetch_ptax_usd(start_date=REF_DATE, end_date=REF_DATE, settings=cfg_bcb)
preview_df(df_ptax, "BCB PTAX — USD fechamento")

PTAX moeda_code: 61
URL (1 dia): https://ptax.bcb.gov.br/ptax_internet/consultaBoletim.do?method=gerarCSVFechamentoMoedaNoPeriodo&ChkMoeda=61&DATAINI=13%2F05%2F2026&DATAFIM=13%2F05%2F2026

BCB PTAX — USD fechamento | shape=(1, 8)


,data,codigo,tipo,moeda,taxa_compra,taxa_venda,paridade_compra,paridade_venda
0,2026-05-13,220,A,USD,4.9112,4.9118,1.0,1.0


## 6. Tesouro Nacional — resultados de leilões

API pública; busca por ano derivado de `DATES`.

In [ ]:
from providers.tesouro import get_resultados, get_resultados_by_dates

resp_ano = get_resultados(ano=int(REF_DATE[:4]))
n_reg = len(resp_ano.get("registros", [])) if isinstance(resp_ano, dict) else 0
print(f"get_resultados(ano={REF_DATE[:4]}): {n_reg} registros")

rows = get_resultados_by_dates(DATES)
preview_list(rows, "get_resultados_by_dates (amostra dict)")
if rows:
    display(pd.DataFrame(rows).head(5))

## 7. SIDRA / IBGE — IPCA (sidrapy)

Snapshot; não usa lista de datas.

In [ ]:
from providers.sidra import SidraIpcaClient

df_sidra = SidraIpcaClient().fetch_table_ipca()
preview_df(df_sidra, "SIDRA IPCA (bruto sidrapy)")

## 8. UpToData — ajustes BMF (arquivos locais)

Requer no `.env`:

- `UPTODATA_PASTA_INTEREST_RATE_BASE`
- `UPTODATA_ARQUIVO_INTEREST_RATE_BASE`

Sem paths válidos retorna DataFrame vazio.

In [6]:
from providers.uptodata import scrap_ajustes_bmf, scrap_ajustes_bmf_for_dates

cfg = settings.uptodata
print("pasta:", cfg.pasta_interest_rate_base or "(não configurado)")
print("prefixo arquivo:", cfg.arquivo_interest_rate_base or "(não configurado)")

df_upto_one = scrap_ajustes_bmf(REF_DATE)
preview_df(df_upto_one, f"UpToData um dia ({REF_DATE})")

df_upto = scrap_ajustes_bmf_for_dates(DATES)
preview_df(df_upto, "UpToData for_dates")

2026-05-18 15:30:04 - providers.uptodata.client - INFO - Found 4 files with prefix Interest_Rate_SettlementPriceFile_Futures_20260513_
2026-05-18 15:30:04 - providers.uptodata.client - INFO - Using newest file: Interest_Rate_SettlementPriceFile_Futures_20260513_3_1.csv


pasta: X:\Interest_Rate\SettlementPrice
prefixo arquivo: Interest_Rate_SettlementPriceFile_Futures_

UpToData um dia (2026-05-13) | shape=(239, 17)


,RptDt,TckrSymb,SctyId,SctySrc,MktIdrCd,ISIN,XprtnDt,AdjstdQt,AdjstdQtTax,AdjstdQtStin,PrvsAdjstdQt,PrvsAdjstdQtTax,PrvsAdjstdQtStin,VartnPts,EqvtVal,AdjstdValCtrct,DataSts
0,2026-05-13,DAPF27,200001817282,8,BVMF,BRBMEFDAP4Y9,2027-01-15,94232.66,9.205,F,94250.83,9.174,U,-18.17,NaN,-34.482163,I
1,2026-05-13,DAPK26,200002398211,8,BVMF,BRBMEFDAP595,2026-05-15,99960.02,5.168,F,99960.04,5.165,U,-0.02,NaN,-0.037955,I
2,2026-05-13,DAPK27,100000164464,8,BVMF,BRBMEFDAP3U9,2027-05-17,92361.69,8.270,F,92393.91,8.232,U,-32.22,NaN,-61.145586,I
3,2026-05-13,DAPK29,200001361511,8,BVMF,BRBMEFDAP4N2,2029-05-15,79625.32,7.945,F,79948.19,7.799,U,-322.87,NaN,-612.727350,I
4,2026-05-13,DAPK31,300000055944,8,BVMF,BRBMEFDAP5E8,2031-05-15,68702.33,7.855,F,69124.27,7.722,U,-421.94,NaN,-800.737690,I


2026-05-18 15:30:04 - providers.uptodata.client - INFO - Processing date: 2026-05-13
2026-05-18 15:30:04 - providers.uptodata.client - INFO - Found 4 files with prefix Interest_Rate_SettlementPriceFile_Futures_20260513_
2026-05-18 15:30:04 - providers.uptodata.client - INFO - Using newest file: Interest_Rate_SettlementPriceFile_Futures_20260513_3_1.csv
2026-05-18 15:30:04 - providers.uptodata.client - INFO - Date 2026-05-13 loaded OK. 239 rows.
2026-05-18 15:30:04 - providers.uptodata.client - INFO - Total rows in combined frame: 239



UpToData for_dates | shape=(239, 17)


,RptDt,TckrSymb,SctyId,SctySrc,MktIdrCd,ISIN,XprtnDt,AdjstdQt,AdjstdQtTax,AdjstdQtStin,PrvsAdjstdQt,PrvsAdjstdQtTax,PrvsAdjstdQtStin,VartnPts,EqvtVal,AdjstdValCtrct,DataSts
0,2026-05-13,DAPF27,200001817282,8,BVMF,BRBMEFDAP4Y9,2027-01-15,94232.66,9.205,F,94250.83,9.174,U,-18.17,NaN,-34.482163,I
1,2026-05-13,DAPK26,200002398211,8,BVMF,BRBMEFDAP595,2026-05-15,99960.02,5.168,F,99960.04,5.165,U,-0.02,NaN,-0.037955,I
2,2026-05-13,DAPK27,100000164464,8,BVMF,BRBMEFDAP3U9,2027-05-17,92361.69,8.270,F,92393.91,8.232,U,-32.22,NaN,-61.145586,I
3,2026-05-13,DAPK29,200001361511,8,BVMF,BRBMEFDAP4N2,2029-05-15,79625.32,7.945,F,79948.19,7.799,U,-322.87,NaN,-612.727350,I
4,2026-05-13,DAPK31,300000055944,8,BVMF,BRBMEFDAP5E8,2031-05-15,68702.33,7.855,F,69124.27,7.722,U,-421.94,NaN,-800.737690,I


## Resumo dos contratos (referência)

| Provider | Contrato em `contracts` |
|----------|-------------------------|
| BCB NegE, UpToData | `DateRangeDataFrameFetcher` |
| BCB SGS / CDI (`fetch_cdi_daily`) | (genérico; mesmo padrão DataFrame por intervalo) |
| BCB PTAX (`fetch_ptax_usd`) | (genérico; CSV fechamento por intervalo) |
| Feriados | `SnapshotDateListFetcher` |
| Tesouro | `DateRangeRecordFetcher` |
| SIDRA | `SidraIpcaProvider` |
| ANBIMA | `AnbimaFeedClient` |